In [ ]:
"""
v3-CV: Stylized Re-Augmentation for ALL Patients
=================================================
For 5-fold Group CV every patient appears in the training set 4/5 folds.
So we augment ALL patients once and save to disk.
Each fold then filters the manifest to its own train patients at runtime.

Run ONCE before CV training cell.
Runtime: ~15–20 minutes on T4 (CPU-bound).
"""

import os
import re
import sys
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

# ─── CONFIG ───────────────────────────────────────────────────
IS_KAGGLE    = os.path.exists('/kaggle')
IS_LIGHTNING = os.path.exists('/teamspace')
workspace    = Path('.').resolve()


def find_first_dir(search_roots, dirname):
    for base in search_roots:
        if not base or not os.path.exists(base):
            continue
        for root, dirs, _ in os.walk(base):
            if dirname in dirs:
                return os.path.join(root, dirname)
    return None


def find_first_file(search_roots, filename):
    for base in search_roots:
        if not base or not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            if filename in files:
                return os.path.join(root, filename)
    return None


if IS_KAGGLE:
    AUG_DIR      = '/kaggle/working/v3_augmented_all'
    AUG_MANIFEST = '/kaggle/working/v3_augmented_all_manifest.csv'
    DATA_SEARCH_ROOTS = ['/kaggle/input']
    ORIG_DATA_DIR  = find_first_dir(DATA_SEARCH_ROOTS, 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations_fixed.csv')
elif IS_LIGHTNING:
    LIGHTNING_WORK_DIR = Path('/teamspace/studios/this_studio')
    if not LIGHTNING_WORK_DIR.exists():
        LIGHTNING_WORK_DIR = workspace
    DATA_SEARCH_ROOTS = [
        str(workspace),
        '/teamspace/datasets',
        '/teamspace/studios/this_studio',
    ]
    ORIG_DATA_DIR   = find_first_dir(DATA_SEARCH_ROOTS, 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations_fixed.csv')
    if ANNOTATIONS_CSV is None:
        ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations.csv')
    AUG_DIR      = str(LIGHTNING_WORK_DIR / 'v3_augmented_all')
    AUG_MANIFEST = str(LIGHTNING_WORK_DIR / 'v3_augmented_all_manifest.csv')
else:
    DATA_SEARCH_ROOTS = [
        str(workspace),
        str(workspace.parent),
        str(workspace / 'Data'),
        str(workspace.parent / 'Data'),
    ]
    ORIG_DATA_DIR   = find_first_dir(DATA_SEARCH_ROOTS, 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations_fixed.csv')
    if ANNOTATIONS_CSV is None:
        ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations.csv')
    AUG_DIR      = str(workspace / 'v3_augmented_all')
    AUG_MANIFEST = str(workspace / 'v3_augmented_all_manifest.csv')

N_VARIANTS = 8   # augmentations per image
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def extract_patient_id(filename):
    for pat in [r'case_(\d+)', r'cys_case_(\d+)']:
        m = re.search(pat, str(filename))
        if m:
            return int(m.group(1))
    return -1


# ─── AUGMENTATION OPERATIONS ──────────────────────────────────
def random_hue_sat_shift(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + random.uniform(-25, 25)) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.55, 1.45), 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.65, 1.35), 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

def random_gamma(img):
    inv = 1.0 / random.uniform(0.65, 1.45)
    table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(img, table)

def random_brightness_contrast(img):
    return cv2.convertScaleAbs(img, alpha=random.uniform(0.75, 1.30),
                               beta=random.uniform(-30, 30))

def random_motion_blur(img):
    if random.random() > 0.4:
        return img
    ksize = random.choice([3, 5, 7])
    angle = random.uniform(0, 180)
    kernel = np.zeros((ksize, ksize))
    kernel[ksize // 2, :] = np.ones(ksize) / ksize
    M = cv2.getRotationMatrix2D((ksize / 2, ksize / 2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (ksize, ksize))
    kernel = kernel / (kernel.sum() + 1e-8)
    return cv2.filter2D(img, -1, kernel)

def random_jpeg_compression(img):
    if random.random() > 0.5:
        return img
    quality = random.randint(35, 80)
    _, encoded = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    return cv2.imdecode(encoded, cv2.IMREAD_COLOR)

def vignette(img):
    if random.random() > 0.35:
        return img
    h, w = img.shape[:2]
    cx = w / 2 + random.uniform(-w * 0.08, w * 0.08)
    cy = h / 2 + random.uniform(-h * 0.08, h * 0.08)
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    max_dist = np.sqrt((w / 2) ** 2 + (h / 2) ** 2)
    strength = random.uniform(0.20, 0.55)
    mask = np.clip(1 - strength * (dist / max_dist) ** 2, 0, 1)
    out = img.astype(np.float32)
    for c in range(3):
        out[:, :, c] *= mask
    return np.clip(out, 0, 255).astype(np.uint8)

def random_geometric(img):
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
    if random.random() > 0.5:
        img = cv2.flip(img, 0)
    if random.random() > 0.5:
        angle = random.uniform(-12, 12)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    return img

def stylized_augment(img):
    img = random_geometric(img)
    img = random_hue_sat_shift(img)
    img = random_gamma(img)
    img = random_brightness_contrast(img)
    img = random_motion_blur(img)
    img = vignette(img)
    img = random_jpeg_compression(img)
    return img


# ─── MAIN ─────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("v3-CV — STYLIZED RE-AUGMENTATION (ALL PATIENTS)")
    print("=" * 60)

    if ORIG_DATA_DIR is None or ANNOTATIONS_CSV is None:
        raise FileNotFoundError("Could not locate EndoscopicBladderTissue or annotations_fixed.csv")

    # Check if manifest already exists — skip if up to date
    if os.path.exists(AUG_MANIFEST):
        existing = pd.read_csv(AUG_MANIFEST)
        print(f"✓ Manifest already exists with {len(existing)} rows at:")
        print(f"  {AUG_MANIFEST}")
        print("  Delete it and re-run this cell if you need to regenerate.")
        return

    os.makedirs(AUG_DIR, exist_ok=True)

    # Load ALL annotations (no patient filtering)
    df = pd.read_csv(ANNOTATIONS_CSV)
    df.columns = df.columns.str.strip()
    filename_col = 'HLY' if 'HLY' in df.columns else df.columns[0]
    df['patient_id'] = df[filename_col].apply(extract_patient_id)

    print(f"Total images to augment: {len(df)}")
    print(f"Unique patients: {df['patient_id'].nunique()}")
    print(f"Output dir: {AUG_DIR}")

    # Build image index
    image_index = {}
    search_root = '/kaggle/input' if IS_KAGGLE else str(Path(ORIG_DATA_DIR).parent)
    for root, _, files in os.walk(search_root):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_index[f.lower()] = os.path.join(root, f)
    print(f"Indexed {len(image_index)} images on disk")

    manifest_rows = []
    skipped = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting all patients"):
        fname = str(row[filename_col]).strip()
        path  = image_index.get(fname.lower())
        if path is None:
            skipped += 1
            continue

        img = cv2.imread(path)
        if img is None:
            skipped += 1
            continue

        h, w = img.shape[:2]
        if max(h, w) > 768:
            s   = 768 / max(h, w)
            img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

        base_name  = os.path.splitext(fname)[0]
        tissue     = str(row.get('tissue type', 'unknown')).strip()
        patient_id = row['patient_id']

        # Save original
        orig_out = os.path.join(AUG_DIR, f"{base_name}_orig.png")
        cv2.imwrite(orig_out, img)
        manifest_rows.append({
            'filename':    f"{base_name}_orig.png",
            'tissue type': tissue,
            'patient_id':  patient_id,
            'is_augmented': False,
            'aug_mode':    'orig',
            'full_path':   orig_out,
        })

        # Generate N_VARIANTS augmented copies
        for v in range(N_VARIANTS):
            aug_img  = stylized_augment(img.copy())
            out_name = f"{base_name}_aug{v}.png"
            out_path = os.path.join(AUG_DIR, out_name)
            cv2.imwrite(out_path, aug_img)
            manifest_rows.append({
                'filename':    out_name,
                'tissue type': tissue,
                'patient_id':  patient_id,
                'is_augmented': True,
                'aug_mode':    f'stylized_v{v}',
                'full_path':   out_path,
            })

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(AUG_MANIFEST, index=False)

    print(f"\n✓ Generated {len(manifest)} total entries ({skipped} skipped)")
    print(f"✓ Manifest saved: {AUG_MANIFEST}")
    print(f"\nPer-class counts (aug+orig):")
    for tt, cnt in manifest['tissue type'].value_counts().items():
        print(f"  {tt}: {cnt}")
    print(f"\nPer-patient counts:")
    for pid, cnt in manifest.groupby('patient_id').size().items():
        print(f"  Patient {pid}: {cnt} images")


if __name__ == '__main__':
    main()
main()


v3-FINAL — STYLIZED RE-AUGMENTATION
Train patients: [4, 5, 7, 8, 10, 13, 14, 16, 21, 22, 23, 24, 25]
Variants per image: 8
Output: /kaggle/working/v3_augmented
Train images to augment: 1211
Indexed 3472 images on disk


Augmenting:   0%|          | 0/1211 [00:00<?, ?it/s]

✓ Generated 10629 total images (30 skipped)
✓ Manifest saved: /kaggle/working/v3_augmented_manifest.csv
Per-class counts:
  LGC: 4320
  HGC: 3321
  NST: 1971
  NTL: 1017


In [5]:
"""
v3-Argmax Baseline: Last-Shot Architecture
================================
- FROZEN DINOv2 + DenseNet121 (no fine-tuning)
- Tiny CLAM (hidden=256, single head)
- Teacher trained on train + heavy stylized augmentation
- 5 students with KL-distillation against teacher's val predictions
- 8-view image-space TTA at inference
- Constrained threshold tuning (HGC recall >= 92%)

Split B: Train=[4,5,7,8,10,13,14,16,21,22,23,24,25]
         Val=[0,1,2,9,12]
         Test=[6,11,17,18]

Prerequisite: Run augment_train_v3.py first to generate /kaggle/working/v3_augmented/
"""

import os
import re
import sys
import copy
import math
import json
import time
import random
import hashlib
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, recall_score, precision_score, f1_score
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════
# 1: CONFIGURATION
# ════════════════════════════════════════════════════════════
IS_KAGGLE = os.path.exists('/kaggle')
IS_LIGHTNING = os.path.exists('/teamspace')
workspace = Path('.').resolve()


def find_first_dir(search_roots, dirname):
    for base in search_roots:
        if not base or not os.path.exists(base):
            continue
        for root, dirs, _ in os.walk(base):
            if dirname in dirs:
                return os.path.join(root, dirname)
    return None


def find_first_file(search_roots, filename):
    for base in search_roots:
        if not base or not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            if filename in files:
                return os.path.join(root, filename)
    return None


if IS_KAGGLE:
    # Augmented data from augment_train_v3.py
    AUG_TRAIN_DIR = '/kaggle/working/v3_augmented_all'
    AUG_TRAIN_MANIFEST = '/kaggle/working/v3_augmented_all_manifest.csv'
    DATA_SEARCH_ROOTS = ['/kaggle/input', '/kaggle/working']
    
    # Original data (for val/test)
    DATASET_ROOT = '/kaggle/input/ebt-22'
    for root, dirs, _ in os.walk('/kaggle/input'):
        if 'EndoscopicBladderTissue' in dirs:
            DATASET_ROOT = root
            break
    ORIG_DATA_DIR = os.path.join(DATASET_ROOT, 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = None
    for root, _, files in os.walk('/kaggle/input'):
        if 'annotations_fixed.csv' in files:
            ANNOTATIONS_CSV = os.path.join(root, 'annotations_fixed.csv')
            break
    
    OUTPUT_DIR = '/kaggle/working/output_5foldcv_frozen'
    CACHE_DIR = '/kaggle/working/feat_cache_v3'
elif IS_LIGHTNING:
    LIGHTNING_WORK_DIR = Path('/teamspace/studios/this_studio')
    if not LIGHTNING_WORK_DIR.exists():
        LIGHTNING_WORK_DIR = workspace
    DATA_SEARCH_ROOTS = [
        str(workspace),
        '/teamspace/datasets',
        '/teamspace/studios/this_studio/ebt-22',
    ]
    ORIG_DATA_DIR = find_first_dir(DATA_SEARCH_ROOTS, 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations_fixed.csv')
    if ANNOTATIONS_CSV is None:
        ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations.csv')
    
    AUG_TRAIN_DIR = str(LIGHTNING_WORK_DIR / 'v3_augmented_all')
    AUG_TRAIN_MANIFEST = str(LIGHTNING_WORK_DIR / 'v3_augmented_all_manifest.csv')
    OUTPUT_DIR = str(LIGHTNING_WORK_DIR / 'output_5foldcv_frozen')
    CACHE_DIR = str(LIGHTNING_WORK_DIR / 'feat_cache_v3')
else:
    # Local workspace
    DATA_SEARCH_ROOTS = [
        str(workspace),
        str(workspace.parent),
        str(workspace / 'Data'),
        str(workspace.parent / 'Data'),
    ]
    ORIG_DATA_DIR = find_first_dir(DATA_SEARCH_ROOTS, 'EndoscopicBladderTissue')
    ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations_fixed.csv')
    if ANNOTATIONS_CSV is None:
        ANNOTATIONS_CSV = find_first_file(DATA_SEARCH_ROOTS, 'annotations.csv')
    
    AUG_TRAIN_DIR = str(workspace / 'v3_augmented_all')
    AUG_TRAIN_MANIFEST = str(workspace / 'v3_augmented_all_manifest.csv')
    OUTPUT_DIR = str(workspace / 'output_v3')
    CACHE_DIR = str(workspace / 'feat_cache_v3')

if ORIG_DATA_DIR is None:
    raise FileNotFoundError(f"Could not find EndoscopicBladderTissue under: {DATA_SEARCH_ROOTS}")
if ANNOTATIONS_CSV is None:
    raise FileNotFoundError(f"Could not find annotations CSV under: {DATA_SEARCH_ROOTS}")

# Splits
TRAIN_PATIENTS = [4, 5, 7, 8, 10, 13, 14, 16, 21, 22, 23, 24, 25]
VAL_PATIENTS   = [0, 1, 2, 9, 12]
TEST_PATIENTS  = [6, 11, 17, 18]

# Class config
NUM_CLASSES = 3
CLASS_NAMES = ['HGC', 'LGC', 'Normal']
LABEL_MAP = {'HGC': 'HGC', 'LGC': 'LGC', 'NST': 'Normal', 'NTL': 'Normal'}
CLASS_TO_IDX = {'HGC': 0, 'LGC': 1, 'Normal': 2}
IDX_TO_CLASS = {0: 'HGC', 1: 'LGC', 2: 'Normal'}
CANCER_CLASSES = {0, 1}

# Image preprocessing
IMAGE_RESIZE = 512
PATCH_SCALES = [96, 128, 192]
PATCH_OUTPUT_SIZE = 224
PATCH_STRIDE_FRAC = 0.5
MIN_TISSUE = 0.40
MAX_BRIGHT = 245
MIN_BRIGHT = 15
MIN_SAT = 10
MIN_FOCUS = 8.0
TOP_QUALITY_FRAC = 0.85
MAX_PATCHES_PER_IMAGE = 60
CLAHE_CLIP = 1.5
CLAHE_GRID = (16, 16)

# Feature extraction
FEAT_BATCH = 128
PATCH_BATCH_TARGET = 512
CACHE_VERSION = 'v3_final'

# ════════════════════════════════════════════════════════════
# BACKBONE FREEZE CONFIG  ← toggle here
# ════════════════════════════════════════════════════════════
FREEZE_BACKBONE = True   # True  = fully frozen (uses feat cache, fast)
                          # False = partial unfreeze last 2 DINOv2 blocks + DenseNet denseblock4
# When FREEZE_BACKBONE=False the feat cache is bypassed for training images.
# ════════════════════════════════════════════════════════════


# CLAM (TINY — anti-overfitting)
MIL_HIDDEN = 256              # ↓ from 512
MIL_DROPOUT_TEACHER = 0.20
MIL_DROPOUT_STUDENTS = [0.30, 0.35, 0.40, 0.45, 0.50]
N_ATT_HEADS = 1               # ↓ from 4
CLAM_K_SAMPLE = 8
FEAT_NOISE_STD = 0.015
FEAT_DROP_P = 0.10

# Training
LR = 1e-4
WD = 1e-4
TEACHER_EPOCHS = 30
STUDENT_EPOCHS = 25
WARMUP_EPOCHS = 3
PATIENCE = 8
GRAD_CLIP = 1.0

# Loss
FOCAL_GAMMA = 2.0
LABEL_SMOOTH = 0.05
BAG_LOSS_W = 1.0
INST_LOSS_W = 0.05
HIER_LOSS_W = 0.10
ORDINAL_LOSS_W = 0.10
DISTILL_W = 0.30              # KL-distillation weight (students)
DISTILL_T = 4.0               # Temperature for distillation

# Class weighting
HGC_WEIGHT_BOOST = 2.5

# Patch limits
MAX_PATCHES_TRAIN = 200
MAX_PATCHES_TEST = 400

# Ensemble
N_STUDENTS = 5
STUDENT_SEEDS = [42, 7, 123, 2024, 999]

# Image-space TTA
N_TTA_VIEWS = 8

# Per-bag standardization (proven safe in v2)
USE_PER_BAG_STD = True

# Device
# Device / GPU compatibility
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CUDA_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
CUDA_CAPABILITY = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
IS_T4 = torch.cuda.is_available() and ('T4' in CUDA_NAME.upper() or CUDA_CAPABILITY == (7, 5))
IS_P100 = torch.cuda.is_available() and ('P100' in CUDA_NAME.upper() or CUDA_CAPABILITY[0] == 6)
AMP_DTYPE = torch.float16
USE_AMP = torch.cuda.is_available()

# Kaggle T4 has 16 GB memory; keep DINOv2 + DenseNet batches comfortably sized.
if IS_T4:
    FEAT_BATCH = min(FEAT_BATCH, 64)
    PATCH_BATCH_TARGET = min(PATCH_BATCH_TARGET, 384)
    CACHE_VERSION = f'{CACHE_VERSION}_t4_dino_dense121'

# P100 fallback: DINOv2/xFormers can hit unsupported CUDA kernels on sm_60.
if IS_P100:
    FEAT_BATCH = min(FEAT_BATCH, 32)
    PATCH_BATCH_TARGET = min(PATCH_BATCH_TARGET, 256)
    MAX_PATCHES_TEST = min(MAX_PATCHES_TEST, 300)

IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# ════════════════════════════════════════════════════════════
# 2: DATA LOADING
# ════════════════════════════════════════════════════════════

def safe_torch_save(obj, path):
    """Write PyTorch objects atomically and avoid the zip writer used by default."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp_path = f"{path}.tmp.{os.getpid()}"
    try:
        with open(tmp_path, 'wb') as f:
            torch.save(obj, f, _use_new_zipfile_serialization=False)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, path)
    finally:
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except OSError:
                pass

def extract_patient_id(filename):
    s = str(filename)
    # Strip aug suffixes first
    s = re.sub(r'_aug\d+\.png$', '', s)
    s = re.sub(r'_orig\.png$', '', s)
    for pat in [r'case_(\d+)', r'cys_case_(\d+)']:
        m = re.search(pat, s)
        if m:
            return int(m.group(1))
    return -1


IMAGE_PATH_INDEX = {}


def scan_for_images():
    global IMAGE_PATH_INDEX
    print("" + "=" * 60)
    print("SCANNING FILESYSTEM")
    print("=" * 60)
    
    search_dirs = []
    if IS_KAGGLE:
        search_dirs = ['/kaggle/input', '/kaggle/working']
    else:
        search_dirs = [str(Path(ORIG_DATA_DIR).parent)]
    
    count = 0
    for d in search_dirs:
        if not os.path.exists(d):
            continue
        for root, _, files in os.walk(d):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    IMAGE_PATH_INDEX[f.lower()] = os.path.join(root, f)
                    count += 1
    print(f"  Indexed {count} images")




class LabNormalizer:
    def __init__(self):
        self.ref = None

    def fit(self, images_bgr):
        stats = {'L': [], 'a': [], 'b': []}
        for img in images_bgr:
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            for i, ch in enumerate(['L', 'a', 'b']):
                stats[ch].append({'m': lab[:, :, i].mean(), 's': lab[:, :, i].std() + 1e-6})
        self.ref = {ch: {'m': np.median([s['m'] for s in stats[ch]]),
                         's': np.median([s['s'] for s in stats[ch]])} for ch in ['L', 'a', 'b']}
        return self

    def transform(self, img_bgr):
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        for i, ch in enumerate(['L', 'a', 'b']):
            c = lab[:, :, i]
            sm, ss = c.mean(), c.std() + 1e-6
            lab[:, :, i] = np.clip((c - sm) * (self.ref[ch]['s'] / ss) + self.ref[ch]['m'], 0, 255)
        lab = lab.astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def load_image(path, norm=None):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    h, w = img.shape[:2]
    s = IMAGE_RESIZE / max(h, w)
    if s != 1:
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    if norm:
        img = norm.transform(img)
    return img


def compute_quality(patch_bgr):
    hsv = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)
    v = hsv[:, :, 2].astype(np.float32)
    s = hsv[:, :, 1].astype(np.float32)
    mask = (v < MAX_BRIGHT) & (v > MIN_BRIGHT) & (s > MIN_SAT)
    tissue_frac = mask.sum() / mask.size
    if tissue_frac < MIN_TISSUE:
        return -1.0
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    focus = cv2.Laplacian(gray, cv2.CV_64F).var()
    if focus < MIN_FOCUS:
        return -1.0
    focus_norm = min(focus / 100.0, 1.0)
    sat_std = s[mask].std() / 50.0 if mask.sum() > 10 else 0
    sat_norm = min(sat_std, 1.0)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = min(edges.sum() / (255.0 * edges.size) * 10, 1.0)
    return 0.3 * tissue_frac + 0.3 * focus_norm + 0.2 * sat_norm + 0.2 * edge_density


def extract_multiscale_patches(image_bgr, max_patches=None):
    if max_patches is None:
        max_patches = MAX_PATCHES_PER_IMAGE
    H, W = image_bgr.shape[:2]
    candidates = []
    cap = max_patches * 3
    for scale in PATCH_SCALES:
        if scale > min(H, W):
            continue
        stride = max(1, int(scale * PATCH_STRIDE_FRAC))
        for y in range(0, H - scale + 1, stride):
            for x in range(0, W - scale + 1, stride):
                if len(candidates) >= cap:
                    break
                crop = image_bgr[y:y + scale, x:x + scale]
                q = compute_quality(crop)
                if q > 0:
                    resized = cv2.resize(crop, (PATCH_OUTPUT_SIZE, PATCH_OUTPUT_SIZE), interpolation=cv2.INTER_AREA)
                    candidates.append((resized, q, scale))
            if len(candidates) >= cap:
                break
    if not candidates:
        min_dim = min(H, W)
        y0, x0 = (H - min_dim) // 2, (W - min_dim) // 2
        crop = image_bgr[y0:y0 + min_dim, x0:x0 + min_dim]
        return [cv2.resize(crop, (PATCH_OUTPUT_SIZE, PATCH_OUTPUT_SIZE))]
    candidates.sort(key=lambda x: x[1], reverse=True)
    n_keep = max(1, int(len(candidates) * TOP_QUALITY_FRAC))
    candidates = candidates[:n_keep][:max_patches]
    return [c[0] for c in candidates]


def fit_normalizer(df):
    print("  Fitting LAB normalizer...")
    samples = []
    sample_paths = df.sample(min(50, len(df))).path.values
    for fp in sample_paths:
        try:
            img = cv2.imread(fp)
            if img is not None:
                h, w = img.shape[:2]
                s = IMAGE_RESIZE / max(h, w)
                if s != 1:
                    img = cv2.resize(img, (int(w * s), int(h * s)))
                samples.append(img)
        except Exception:
            pass
    if not samples:
        return None
    norm = LabNormalizer().fit(samples)
    print(f"  ✓ Normalizer fitted on {len(samples)} images")
    return norm


# ════════════════════════════════════════════════════════════
# 4: FEATURE EXTRACTION (FROZEN BACKBONES)
# ════════════════════════════════════════════════════════════
dino_model = None
dense_model = None
feat_dim = 0


def load_backbones():
    global dino_model, dense_model, feat_dim, IMNET_MEAN, IMNET_STD
    mode = "FROZEN" if FREEZE_BACKBONE else "PARTIAL-UNFREEZE (DINOv2 last 2 blocks + DenseBlock4)"
    print("" + "=" * 60)
    print(f"LOADING BACKBONES — {mode}")
    print("=" * 60)

    IMNET_MEAN = IMNET_MEAN.to(DEVICE)
    IMNET_STD  = IMNET_STD.to(DEVICE)

    dino_dim = 0
    print("  Loading DINOv2...")
    try:
        dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        dino_model.to(DEVICE)
        if FREEZE_BACKBONE:
            dino_model.eval()
            for p in dino_model.parameters():
                p.requires_grad = False
            print(f"  ✓ dinov2_vitb14 — FULLY FROZEN, dim=768")
        else:
            for p in dino_model.parameters():
                p.requires_grad = False
            for blk in list(dino_model.blocks[-2:]):
                for p in blk.parameters():
                    p.requires_grad = True
            for p in dino_model.norm.parameters():
                p.requires_grad = True
            trainable = sum(p.numel() for p in dino_model.parameters() if p.requires_grad)
            print(f"  ✓ dinov2_vitb14 — PARTIAL UNFREEZE (last 2 blocks), trainable={trainable:,}")
        dino_dim = 768
    except Exception as e:
        print(f"  ⚠ DINOv2 failed: {e}")

    densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    dense_dim = densenet.classifier.in_features
    densenet.classifier = nn.Identity()
    dense_model = densenet.to(DEVICE)
    if FREEZE_BACKBONE:
        dense_model.eval()
        for p in dense_model.parameters():
            p.requires_grad = False
        print(f"  ✓ DenseNet121 — FULLY FROZEN, dim={dense_dim}")
    else:
        for p in dense_model.parameters():
            p.requires_grad = False
        for p in dense_model.features.denseblock4.parameters():
            p.requires_grad = True
        for p in dense_model.features.norm5.parameters():
            p.requires_grad = True
        trainable = sum(p.numel() for p in dense_model.parameters() if p.requires_grad)
        print(f"  ✓ DenseNet121 — PARTIAL UNFREEZE (denseblock4+norm5), trainable={trainable:,}")

    feat_dim = (dino_dim if dino_model else 0) + dense_dim
    print(f"  Total feature dim: {feat_dim}")
    return feat_dim

def bgr_to_tensor(patch_bgr):
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0


def _get_cache_key(path, tta_idx=None):
    parts = f"{path}|{IMAGE_RESIZE}|{PATCH_SCALES}|{MAX_PATCHES_PER_IMAGE}|{CACHE_VERSION}"
    if tta_idx is not None:
        parts += f"|tta{tta_idx}"
    return hashlib.md5(parts.encode()).hexdigest()


@torch.inference_mode()
def extract_dual_features(tensor_list):
    if not tensor_list:
        return torch.empty((0, feat_dim), dtype=torch.float16)
    all_feats = []
    for i in range(0, len(tensor_list), FEAT_BATCH):
        batch = torch.stack(tensor_list[i:i + FEAT_BATCH]).to(DEVICE, non_blocking=True)
        batch_norm = (batch - IMNET_MEAN) / IMNET_STD
        parts = []
        with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
            if dino_model is not None:
                dino_out = dino_model(batch_norm)
                if isinstance(dino_out, dict):
                    dino_feats = dino_out.get('x_norm_clstoken', next(iter(dino_out.values())))
                else:
                    dino_feats = dino_out
                if dino_feats.dim() > 2:
                    dino_feats = dino_feats[:, 0, :]
                parts.append(dino_feats.float().cpu())
            parts.append(dense_model(batch_norm).float().cpu())
        all_feats.append(torch.cat(parts, dim=1))
    return torch.cat(all_feats, 0).half()


def extract_features_for_split(df, desc="Extracting", norm=None, use_cache=True):
    os.makedirs(CACHE_DIR, exist_ok=True)
    results, skipped, cache_hits = [], 0, 0
    pending_tensors, pending_meta = [], []
    
    def flush():
        nonlocal pending_tensors, pending_meta
        if not pending_tensors:
            return
        feats = extract_dual_features(pending_tensors)
        start = 0
        for meta in pending_meta:
            n = meta['n_patches']
            block = feats[start:start + n]
            start += n
            if meta['cache_path']:
                try:
                    torch.save({'features': block}, meta['cache_path'])
                except Exception:
                    pass
            results.append({
                'features': block,
                'label': meta['label'],
                'label_name': meta['label_name'],
                'patient': meta['patient'],
                'path': meta['path'],
                'n_patches': block.shape[0],
            })
        pending_tensors.clear()
        pending_meta.clear()
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        cache_path = None
        if use_cache:
            cache_key = _get_cache_key(str(row.path))
            cache_path = os.path.join(CACHE_DIR, f"{cache_key}.pt")
            if os.path.exists(cache_path):
                try:
                    cached = torch.load(cache_path, map_location='cpu', weights_only=False)
                    results.append({
                        'features': cached['features'],
                        'label': int(row.target),
                        'label_name': row.label,
                        'patient': int(row.patient),
                        'path': row.path,
                        'n_patches': cached['features'].shape[0],
                    })
                    cache_hits += 1
                    continue
                except Exception:
                    pass
        
        try:
            img = load_image(str(row.path), norm)
        except Exception:
            skipped += 1
            continue
        
        patches = extract_multiscale_patches(img)
        if not patches:
            skipped += 1
            continue
        
        tensors = [bgr_to_tensor(p) for p in patches]
        if len(tensors) > MAX_PATCHES_PER_IMAGE:
            idx = random.sample(range(len(tensors)), MAX_PATCHES_PER_IMAGE)
            tensors = [tensors[i] for i in sorted(idx)]
        
        pending_tensors.extend(tensors)
        pending_meta.append({
            'label': int(row.target),
            'label_name': row.label,
            'patient': int(row.patient),
            'path': str(row.path),
            'n_patches': len(tensors),
            'cache_path': cache_path,
        })
        
        if len(pending_tensors) >= PATCH_BATCH_TARGET:
            flush()
    
    flush()
    
    if skipped:
        print(f"  ⚠ Skipped {skipped}")
    if cache_hits:
        print(f"  ⚡ Cache hits: {cache_hits}/{len(df)}")
    print(f"  ✓ Extracted {len(results)} bags")
    return results


# ════════════════════════════════════════════════════════════
# 5: TINY CLAM MODEL
# ════════════════════════════════════════════════════════════
class TinyCLAM(nn.Module):
    """
    Minimal CLAM: hidden=256, single attention head, no adapter.
    Designed to NOT memorize on N=22 patients.
    """
    def __init__(self, feat_dim_in, hidden=MIL_HIDDEN, n_classes=NUM_CLASSES,
                 dropout=0.25, k_sample=CLAM_K_SAMPLE):
        super().__init__()
        self.n_classes = n_classes
        self.k_sample = k_sample
        self.feat_noise = FEAT_NOISE_STD
        self.feat_drop = nn.Dropout(FEAT_DROP_P)
        
        # Single linear encoder (no adapter)
        self.fc = nn.Sequential(
            nn.Linear(feat_dim_in, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        # Single gated attention head per class
        self.att_branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 4),
                nn.Tanh(),
            ) for _ in range(n_classes)
        ])
        self.gate_branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 4),
                nn.Sigmoid(),
            ) for _ in range(n_classes)
        ])
        self.att_combine = nn.ModuleList([
            nn.Linear(hidden // 4, 1) for _ in range(n_classes)
        ])
        
        # Per-class instance classifiers
        self.inst_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, 64), nn.GELU(),
                nn.Linear(64, 2)
            ) for _ in range(n_classes)
        ])
        
        # Per-class bag classifiers
        self.bag_classifiers = nn.ModuleList([
            nn.Linear(hidden, 1) for _ in range(n_classes)
        ])

    def _inst_loss(self, scores, h, classifier, k):
        N = scores.shape[0]
        k = min(k, N // 2, 8)
        if k < 1:
            return torch.tensor(0.0, device=h.device)
        top_idx = torch.topk(scores, k).indices
        bot_idx = torch.topk(scores, k, largest=False).indices
        feats = torch.cat([h[top_idx], h[bot_idx]], dim=0)
        labels = torch.cat([torch.ones(k, dtype=torch.long), torch.zeros(k, dtype=torch.long)]).to(h.device)
        return F.cross_entropy(classifier(feats), labels)

    def forward(self, x, label=None):
        input_dtype = x.dtype
        
        # Per-bag standardization
        if USE_PER_BAG_STD:
            x_f = x.float()
            x = ((x_f - x_f.mean(0, keepdim=True)) / (x_f.std(0, keepdim=True) + 1e-6)).to(input_dtype)
        
        if self.training:
            x = x.float()
            x = x + torch.randn_like(x) * self.feat_noise
            x = x.to(input_dtype)
            x = self.feat_drop(x)
        
        h = self.fc(x)  # (N, hidden)
        
        logits = []
        total_inst = torch.tensor(0.0, device=x.device)
        
        for c in range(self.n_classes):
            a = self.att_branches[c](h)
            g = self.gate_branches[c](h)
            scores = self.att_combine[c](a * g).squeeze(-1)
            weights = F.softmax(scores, dim=0)
            bag = torch.sum(weights.unsqueeze(-1) * h, dim=0)
            logits.append(self.bag_classifiers[c](bag))
            
            if self.training and label is not None and label.item() == c:
                total_inst += self._inst_loss(scores.detach(), h, self.inst_classifiers[c], self.k_sample)
        
        return {'logits': torch.cat(logits), 'inst_loss': total_inst}


# ════════════════════════════════════════════════════════════
# 6: LOSSES
# ════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def hierarchical_loss(logits, label):
    cancer_score = logits[0] + logits[1]
    normal_score = logits[2]
    binary_logit = torch.stack([cancer_score, normal_score]).unsqueeze(0)
    binary_target = torch.tensor([0] if label.item() in CANCER_CLASSES else [1],
                                 dtype=torch.long, device=label.device)
    return F.cross_entropy(binary_logit, binary_target)


def ordinal_loss(logits, label):
    probs = F.softmax(logits, dim=0)
    severity = torch.arange(NUM_CLASSES, dtype=torch.float32, device=logits.device)
    pred_sev = (probs * severity).sum()
    true_sev = severity[label]
    return (pred_sev - true_sev) ** 2


def supervised_loss(output, label, class_weights=None, focal_criterion=None):
    logits = output['logits'].unsqueeze(0)
    target = label.unsqueeze(0)
    bag_loss = focal_criterion(logits, target) if focal_criterion else \
        F.cross_entropy(logits, target, weight=class_weights, label_smoothing=LABEL_SMOOTH)
    hier = hierarchical_loss(output['logits'], label)
    ordi = ordinal_loss(output['logits'], label)
    return BAG_LOSS_W * bag_loss + HIER_LOSS_W * hier + ORDINAL_LOSS_W * ordi + INST_LOSS_W * output['inst_loss']


def kl_distill_loss(student_logits, teacher_logits, T=DISTILL_T):
    """KL divergence between student and teacher (softened)."""
    student_log_probs = F.log_softmax(student_logits / T, dim=0)
    teacher_probs = F.softmax(teacher_logits / T, dim=0)
    return F.kl_div(student_log_probs.unsqueeze(0), teacher_probs.unsqueeze(0),
                    reduction='batchmean') * (T * T)


# ════════════════════════════════════════════════════════════
# 7: TRAINING UTILITIES
# ════════════════════════════════════════════════════════════
def compute_class_weights(data_list):
    counts = Counter(d['label'] for d in data_list)
    total = sum(counts.values())
    weights = []
    for c in range(NUM_CLASSES):
        w = total / (NUM_CLASSES * max(counts.get(c, 0), 1))
        if c == CLASS_TO_IDX['HGC']:
            w *= HGC_WEIGHT_BOOST
        weights.append(w)
    wt = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"Class weights (HGC ×{HGC_WEIGHT_BOOST}):")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"    {cls}: count={counts.get(i,0)} → w={weights[i]:.3f}")
    return wt


def class_balanced_sample(data_list):
    by_class = defaultdict(list)
    for d in data_list:
        by_class[d['label']].append(d)
    max_count = max(len(v) for v in by_class.values())
    balanced = []
    for cls, items in by_class.items():
        balanced.extend(items)
        if len(items) < max_count:
            balanced.extend(random.choices(items, k=max_count - len(items)))
    random.shuffle(balanced)
    return balanced


def warmup_cosine(optimizer, warmup, total):
    def lr_lambda(epoch):
        if epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / max(total - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ════════════════════════════════════════════════════════════
# 8: TEACHER TRAINING
# ════════════════════════════════════════════════════════════
def train_teacher(train_data, val_data, class_weights):
    print("" + "=" * 60)
    print("TRAINING TEACHER")
    print("=" * 60)
    
    set_seed(42)
    teacher = TinyCLAM(
        feat_dim_in=feat_dim,
        hidden=MIL_HIDDEN,
        dropout=MIL_DROPOUT_TEACHER,
    ).to(DEVICE)
    
    n_params = sum(p.numel() for p in teacher.parameters() if p.requires_grad)
    print(f"  Teacher params: {n_params:,}")
    
    focal = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights, label_smoothing=LABEL_SMOOTH).to(DEVICE)
    optim = torch.optim.AdamW(teacher.parameters(), lr=LR, weight_decay=WD)
    sched = warmup_cosine(optim, WARMUP_EPOCHS, TEACHER_EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    
    best_val_loss = float('inf')
    best_state = None
    patience = 0
    
    for epoch in range(TEACHER_EPOCHS):
        teacher.train()
        epoch_data = class_balanced_sample(train_data)
        
        running = 0.0
        correct = 0
        total = 0
        
        for sample in epoch_data:
            feats = sample['features'].to(DEVICE)
            if feats.shape[0] > MAX_PATCHES_TRAIN:
                idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TRAIN]
                feats = feats[idx]
            label = torch.tensor(sample['label'], dtype=torch.long, device=DEVICE)
            
            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                out = teacher(feats, label=label)
                loss = supervised_loss(out, label, class_weights, focal)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(teacher.parameters(), GRAD_CLIP)
            scaler.step(optim)
            scaler.update()
            
            running += loss.item()
            correct += (out['logits'].argmax().item() == sample['label'])
            total += 1
        
        sched.step()
        train_loss = running / max(total, 1)
        train_acc = correct / max(total, 1)
        
        # Val
        teacher.eval()
        v_loss = 0.0
        v_correct = 0
        with torch.no_grad():
            for sample in val_data:
                feats = sample['features'].to(DEVICE)
                if feats.shape[0] > MAX_PATCHES_TEST:
                    idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
                    feats = feats[idx]
                label = torch.tensor(sample['label'], dtype=torch.long, device=DEVICE)
                with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                    out = teacher(feats)
                loss = F.cross_entropy(out['logits'].float().unsqueeze(0),
                                       label.unsqueeze(0), weight=class_weights)
                v_loss += loss.item()
                if out['logits'].argmax().item() == sample['label']:
                    v_correct += 1
        v_loss /= len(val_data)
        v_acc = v_correct / len(val_data)
        
        improved = ""
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = copy.deepcopy(teacher.state_dict())
            patience = 0
            improved = " *"
        else:
            patience += 1
        
        if epoch < 3 or (epoch + 1) % 3 == 0 or improved or epoch == TEACHER_EPOCHS - 1:
            print(f"    Epoch {epoch+1:3d}: train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
                  f"val_loss={v_loss:.4f} val_acc={v_acc:.3f}{improved}")
        
        if patience >= PATIENCE:
            print(f"    Early stop at epoch {epoch+1}")
            break
    
    if best_state:
        teacher.load_state_dict(best_state)
        print(f"    ✓ Restored teacher (val_loss={best_val_loss:.4f})")
    
    # SAVE IMMEDIATELY
    teacher_path = os.path.join(OUTPUT_DIR, 'teacher.pt')
    safe_torch_save(teacher.state_dict(), teacher_path)
    print(f"    [OK] Teacher checkpoint saved: {teacher_path}")
    return teacher


# ════════════════════════════════════════════════════════════
# 9: STUDENT TRAINING (with teacher distillation on val)
# ════════════════════════════════════════════════════════════
@torch.no_grad()
def compute_teacher_val_logits(teacher, val_data):
    """Pre-compute teacher's logits on val for distillation targets."""
    teacher.eval()
    teacher_logits = []
    for sample in val_data:
        feats = sample['features'].to(DEVICE)
        if feats.shape[0] > MAX_PATCHES_TEST:
            idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
            feats = feats[idx]
        with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
            out = teacher(feats)
        teacher_logits.append(out['logits'].float().detach().cpu())
    return teacher_logits


def train_student(student_idx, train_data, val_data, val_teacher_logits,
                  class_weights):
    seed = STUDENT_SEEDS[student_idx]
    dropout = MIL_DROPOUT_STUDENTS[student_idx]
    
    print(f"── Student {student_idx+1}/{N_STUDENTS}: seed={seed}, dropout={dropout} ──")
    set_seed(seed)
    
    student = TinyCLAM(
        feat_dim_in=feat_dim,
        hidden=MIL_HIDDEN,
        dropout=dropout,
    ).to(DEVICE)
    
    focal = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights, label_smoothing=LABEL_SMOOTH).to(DEVICE)
    optim = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WD)
    sched = warmup_cosine(optim, WARMUP_EPOCHS, STUDENT_EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    
    best_val_loss = float('inf')
    best_state = None
    patience = 0
    
    for epoch in range(STUDENT_EPOCHS):
        student.train()
        epoch_data = class_balanced_sample(train_data)
        
        # Mix in val samples (with teacher logits) for distillation
        # We iterate train data with supervised loss + sample val for distill
        running_sup = 0.0
        running_distill = 0.0
        correct = 0
        total = 0
        
        for sample in epoch_data:
            feats = sample['features'].to(DEVICE)
            if feats.shape[0] > MAX_PATCHES_TRAIN:
                idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TRAIN]
                feats = feats[idx]
            label = torch.tensor(sample['label'], dtype=torch.long, device=DEVICE)
            
            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                out = student(feats, label=label)
                sup_loss = supervised_loss(out, label, class_weights, focal)
            
            scaler.scale(sup_loss).backward()
            running_sup += sup_loss.item()
            
            # Distillation step: forward on a random val sample, KL against teacher
            if random.random() < 0.5:  # distill on ~half of train steps
                val_idx = random.randrange(len(val_data))
                val_sample = val_data[val_idx]
                val_feats = val_sample['features'].to(DEVICE)
                if val_feats.shape[0] > MAX_PATCHES_TRAIN:
                    idx = torch.randperm(val_feats.shape[0])[:MAX_PATCHES_TRAIN]
                    val_feats = val_feats[idx]
                t_logits = val_teacher_logits[val_idx].to(DEVICE)
                
                with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                    val_out = student(val_feats)
                    distill_loss = DISTILL_W * kl_distill_loss(val_out['logits'].float(), t_logits)
                
                scaler.scale(distill_loss).backward()
                running_distill += distill_loss.item()
            
            scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
            scaler.step(optim)
            scaler.update()
            
            correct += (out['logits'].argmax().item() == sample['label'])
            total += 1
        
        sched.step()
        train_loss = running_sup / max(total, 1)
        distill_avg = running_distill / max(total, 1)
        train_acc = correct / max(total, 1)
        
        # Val eval
        student.eval()
        v_loss = 0.0
        v_correct = 0
        with torch.no_grad():
            for sample in val_data:
                feats = sample['features'].to(DEVICE)
                if feats.shape[0] > MAX_PATCHES_TEST:
                    idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
                    feats = feats[idx]
                label = torch.tensor(sample['label'], dtype=torch.long, device=DEVICE)
                with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                    out = student(feats)
                loss = F.cross_entropy(out['logits'].float().unsqueeze(0),
                                       label.unsqueeze(0), weight=class_weights)
                v_loss += loss.item()
                if out['logits'].argmax().item() == sample['label']:
                    v_correct += 1
        v_loss /= len(val_data)
        v_acc = v_correct / len(val_data)
        
        improved = ""
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = copy.deepcopy(student.state_dict())
            patience = 0
            improved = " *"
        else:
            patience += 1
        
        if epoch < 3 or (epoch + 1) % 3 == 0 or improved or epoch == STUDENT_EPOCHS - 1:
            print(f"    Epoch {epoch+1:3d}: sup={train_loss:.4f} distill={distill_avg:.4f} "
                  f"train_acc={train_acc:.3f} | val_loss={v_loss:.4f} val_acc={v_acc:.3f}{improved}")
        
        if patience >= PATIENCE:
            print(f"    Early stop at epoch {epoch+1}")
            break
    
    if best_state:
        student.load_state_dict(best_state)
        print(f"    ✓ Restored student (val_loss={best_val_loss:.4f})")
    
    # SAVE IMMEDIATELY
    s_path = os.path.join(OUTPUT_DIR, f'student_{student_idx}.pt')
    safe_torch_save(student.state_dict(), s_path)
    print(f"    [OK] Student checkpoint saved: {s_path}")
    return student


# ════════════════════════════════════════════════════════════
# 10: IMAGE-SPACE TTA + INFERENCE
# ════════════════════════════════════════════════════════════
def tta_transform(img_bgr, tta_idx):
    """Apply one of 8 TTA transformations to image."""
    if tta_idx == 0:
        return img_bgr  # identity
    elif tta_idx == 1:
        return cv2.flip(img_bgr, 1)  # h-flip
    elif tta_idx == 2:
        return cv2.flip(img_bgr, 0)  # v-flip
    elif tta_idx == 3:
        return cv2.flip(cv2.flip(img_bgr, 1), 0)  # h+v
    elif tta_idx == 4:
        return cv2.rotate(img_bgr, cv2.ROTATE_90_CLOCKWISE)
    elif tta_idx == 5:
        return cv2.rotate(img_bgr, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif tta_idx == 6:
        # Slight brightness up
        return cv2.convertScaleAbs(img_bgr, alpha=1.1, beta=10)
    elif tta_idx == 7:
        # Slight brightness down
        return cv2.convertScaleAbs(img_bgr, alpha=0.9, beta=-10)
    return img_bgr


def extract_features_with_tta(df, norm, desc="TTA features"):
    """Extract features for all images × N_TTA_VIEWS, with caching."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    results = []  # results[i] = list of N_TTA_VIEWS feature tensors
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        per_image_features = []
        
        # Load image once
        try:
            img_orig = load_image(str(row.path), norm)
        except Exception:
            results.append(None)
            continue
        
        for tta_idx in range(N_TTA_VIEWS):
            cache_key = _get_cache_key(str(row.path), tta_idx=tta_idx)
            cache_path = os.path.join(CACHE_DIR, f"{cache_key}.pt")
            
            if os.path.exists(cache_path):
                try:
                    cached = torch.load(cache_path, map_location='cpu', weights_only=False)
                    per_image_features.append(cached['features'])
                    continue
                except Exception:
                    pass
            
            # Apply TTA
            img_tta = tta_transform(img_orig, tta_idx)
            patches = extract_multiscale_patches(img_tta)
            if not patches:
                per_image_features.append(None)
                continue
            tensors = [bgr_to_tensor(p) for p in patches]
            if len(tensors) > MAX_PATCHES_PER_IMAGE:
                idx = random.sample(range(len(tensors)), MAX_PATCHES_PER_IMAGE)
                tensors = [tensors[i] for i in sorted(idx)]
            
            feats = extract_dual_features(tensors)
            try:
                torch.save({'features': feats}, cache_path)
            except Exception:
                pass
            per_image_features.append(feats)
        
        results.append({
            'tta_features': per_image_features,
            'label': int(row.target),
            'patient': int(row.patient),
            'path': str(row.path),
        })
    
    print(f"  ✓ Extracted TTA features for {len(results)} images × {N_TTA_VIEWS} views")
    return results


@torch.no_grad()
def predict_with_tta_ensemble(students, tta_data_item):
    """For one image: run all students × all TTA views, average softmax."""
    if tta_data_item is None:
        return np.array([1/3, 1/3, 1/3])
    
    all_probs = []
    for tta_feats in tta_data_item['tta_features']:
        if tta_feats is None or tta_feats.shape[0] == 0:
            continue
        feats = tta_feats.to(DEVICE)
        if feats.shape[0] > MAX_PATCHES_TEST:
            idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
            feats = feats[idx]
        
        for student in students:
            student.eval()
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                out = student(feats)
            probs = F.softmax(out['logits'].float(), dim=0).cpu().numpy()
            all_probs.append(probs)
    
    if not all_probs:
        return np.array([1/3, 1/3, 1/3])
    avg_probs = np.mean(all_probs, axis=0)
    avg_probs = avg_probs / (avg_probs.sum() + 1e-8)
    return avg_probs


def evaluate_full(students, tta_data, desc="Evaluating"):
    all_preds = []
    all_labels = []
    all_probs = []
    for item in tqdm(tta_data, desc=desc):
        if item is None:
            continue
        probs = predict_with_tta_ensemble(students, item)
        all_preds.append(int(np.argmax(probs)))
        all_labels.append(item['label'])
        all_probs.append(probs)
    return all_preds, all_labels, all_probs


# ════════════════════════════════════════════════════════════
# 11: CONSTRAINED THRESHOLD TUNING
# ════════════════════════════════════════════════════════════
# def tune_constrained_thresholds(val_labels, val_probs, hgc_recall_target=0.92):
#     print("" + "=" * 60)
#     print(f"  CONSTRAINED THRESHOLD TUNING (HGC recall ≥ {hgc_recall_target*100:.0f}%)")
#     print("=" * 60)
    
#     val_labels = np.array(val_labels)
#     val_probs = np.array(val_probs)
#     hgc_idx = CLASS_TO_IDX['HGC']
#     lgc_idx = CLASS_TO_IDX['LGC']
#     hgc_true = (val_labels == hgc_idx)
#     n_hgc = hgc_true.sum()
    
#     base_preds = np.argmax(val_probs, axis=1)
#     base_bal = balanced_accuracy_score(val_labels, base_preds)
#     base_hgc_r = ((base_preds == hgc_idx) & hgc_true).sum() / max(n_hgc, 1)
#     print(f"  Argmax baseline: bal_acc={base_bal*100:.1f}%, HGC_recall={base_hgc_r*100:.1f}%")
    
#     best = None
#     best_bal = -1
    
#     for hgc_t in np.arange(0.15, 0.55, 0.025):
#         for lgc_t in np.arange(0.25, 0.55, 0.025):
#             preds = []
#             for p in val_probs:
#                 if p[hgc_idx] > hgc_t:
#                     preds.append(hgc_idx)
#                 elif p[lgc_idx] > lgc_t:
#                     preds.append(lgc_idx)
#                 else:
#                     preds.append(int(np.argmax(p)))
#             preds = np.array(preds)
#             hgc_r = ((preds == hgc_idx) & hgc_true).sum() / max(n_hgc, 1)
#             if hgc_r < hgc_recall_target:
#                 continue
#             bal = balanced_accuracy_score(val_labels, preds)
#             if bal > best_bal:
#                 best_bal = bal
#                 hgc_p = ((preds == hgc_idx) & hgc_true).sum() / max((preds == hgc_idx).sum(), 1)
#                 best = {
#                     'hgc_threshold': float(hgc_t),
#                     'lgc_threshold': float(lgc_t),
#                     'bal_acc': bal,
#                     'hgc_recall': hgc_r,
#                     'hgc_precision': hgc_p,
#                     'accuracy': accuracy_score(val_labels, preds),
#                 }
    
#     if best:
#         print(f"  ✓ Best: HGC_t={best['hgc_threshold']:.3f}, LGC_t={best['lgc_threshold']:.3f}")
#         print(f"    val: bal_acc={best['bal_acc']*100:.1f}%, "
#               f"HGC_R={best['hgc_recall']*100:.1f}%, HGC_P={best['hgc_precision']*100:.1f}%")
#         return {hgc_idx: best['hgc_threshold'], lgc_idx: best['lgc_threshold']}, best
    
#     print(f"  ⚠ Constraint not satisfiable. Falling back to lower target...")
#     if hgc_recall_target > 0.85:
#         return tune_constrained_thresholds(val_labels, val_probs, hgc_recall_target=0.85)
#     return {hgc_idx: 0.3, lgc_idx: 0.4}, None


# def apply_thresholds(probs_list, thresholds):
#     preds = []
#     hgc_idx = CLASS_TO_IDX['HGC']
#     lgc_idx = CLASS_TO_IDX['LGC']
#     hgc_t = thresholds.get(hgc_idx, 0.5)
#     lgc_t = thresholds.get(lgc_idx, 0.5)
#     for p in probs_list:
#         if p[hgc_idx] > hgc_t:
#             preds.append(hgc_idx)
#         elif p[lgc_idx] > lgc_t:
#             preds.append(lgc_idx)
#         else:
#             preds.append(int(np.argmax(p)))
#     return preds


# ════════════════════════════════════════════════════════════
# 12: REPORTING
# ════════════════════════════════════════════════════════════
def print_results(preds, labels, name):
    preds = np.array(preds)
    labels = np.array(labels)
    acc = accuracy_score(labels, preds)
    bal = balanced_accuracy_score(labels, preds)
    print(f"{'=' * 60}{name}{'=' * 60}")
    print(f"  Accuracy:          {acc*100:.2f}%")
    print(f"  Balanced Accuracy: {bal*100:.2f}%")
    print(f"{classification_report(labels, preds, target_names=CLASS_NAMES, digits=4, zero_division=0)}")
    hgc_idx = CLASS_TO_IDX['HGC']
    hgc_true = (labels == hgc_idx)
    hgc_pred = (preds == hgc_idx)
    hgc_tp = (hgc_pred & hgc_true).sum()
    hgc_r = hgc_tp / max(hgc_true.sum(), 1)
    hgc_p = hgc_tp / max(hgc_pred.sum(), 1)
    print(f"  HGC: Recall={hgc_r*100:.2f}%, Precision={hgc_p*100:.2f}%")
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    print(f"Confusion Matrix:{'':>10} {'  '.join(f'{c:>8s}' for c in CLASS_NAMES)}")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"  {cls:>10} {'  '.join(f'{cm[i,j]:8d}' for j in range(NUM_CLASSES))}")
    target_acc = "✅" if acc >= 0.90 else "❌"
    target_hgc = "✅" if hgc_r >= 0.95 else "❌"
    print(f"Target acc≥90%:    {target_acc} ({acc*100:.1f}%)")
    print(f"  Target HGC R≥95%:  {target_hgc} ({hgc_r*100:.1f}%)")
    return {'accuracy': acc, 'balanced_accuracy': bal,
            'hgc_recall': hgc_r, 'hgc_precision': hgc_p}


def save_cm(preds, labels, fname, title):
    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.close()


# ════════════════════════════════════════════════════════════
# 13: MAIN
# ════════════════════════════════════════════════════════════


def load_data_for_fold(fold_idx, train_pids, test_pids, val_pids=None):
    """Load data for a specific CV fold.
    
    train_pids: patients used for training this fold
    test_pids:  patients held out as test for this fold
    val_pids:   if None, use a 20% random sample of train_pids as val
    """
    scan_for_images()

    print("" + "=" * 60)
    print(f"FOLD {fold_idx} — LOADING DATA")
    print("=" * 60)
    print(f"  Train patients: {sorted(train_pids)}")
    print(f"  Test  patients: {sorted(test_pids)}")

    # ── Augmented train data from manifest ──
    if not os.path.exists(AUG_TRAIN_MANIFEST):
        raise FileNotFoundError(f"v3 augmented manifest not found at {AUG_TRAIN_MANIFEST}. "
                                 "Run Cell 0 first to generate it.")
    full_manifest = pd.read_csv(AUG_TRAIN_MANIFEST)
    full_manifest['label'] = full_manifest['tissue type'].map(LABEL_MAP)
    full_manifest['target'] = full_manifest['label'].map(CLASS_TO_IDX)
    full_manifest['patient'] = full_manifest['patient_id']
    full_manifest['path'] = full_manifest['full_path']

    train_df = full_manifest[full_manifest['patient_id'].isin(train_pids)].copy()

    # ── Original annotations for val / test ──
    df_orig = pd.read_csv(ANNOTATIONS_CSV)
    df_orig.columns = df_orig.columns.str.strip()
    df_orig['patient_id'] = df_orig['HLY'].apply(extract_patient_id)
    df_orig['label'] = df_orig['tissue type'].map(LABEL_MAP)
    df_orig['target'] = df_orig['label'].map(CLASS_TO_IDX)
    df_orig['patient'] = df_orig['patient_id']
    df_orig['is_augmented'] = False
    df_orig['aug_mode'] = 'orig'

    def resolve_orig(row):
        fname = str(row['HLY']).strip()
        return IMAGE_PATH_INDEX.get(fname.lower())

    df_orig['path'] = df_orig.apply(resolve_orig, axis=1)
    df_orig = df_orig[df_orig['path'].notna()].copy()

    test_all  = df_orig[df_orig['patient_id'].isin(test_pids)].copy()
    test_df = test_all  # keep all imaging types in test

    # Val: take WLI-only from original non-test patients that are in train_pids
    # Use a heldout subset of train patients for validation
    if val_pids is not None:
        val_all = df_orig[df_orig['patient_id'].isin(val_pids)].copy()
        print(f"  Val patients:   {sorted(val_pids)}")
    else:
        # Take 1 patient from train_pids as val (smallest patient by image count)
        pid_counts = df_orig[df_orig['patient_id'].isin(train_pids)].groupby('patient_id').size()
        val_pid = [pid_counts.idxmin()]
        val_all = df_orig[df_orig['patient_id'].isin(val_pid)].copy()
        train_df = train_df[~train_df['patient_id'].isin(val_pid)].copy()
        print(f"  Val patients (auto, 1 smallest train patient): {val_pid}")

    val_df = val_all[val_all['imaging type'].astype(str).str.upper().eq('WLI')].copy()
    print(f"  Val WLI-only filter: kept {len(val_df)} / {len(val_all)}")

    # Verify disjointness
    train_pids_actual = set(train_df['patient_id'].unique())
    val_pids_actual   = set(val_df['patient_id'].unique())
    test_pids_actual  = set(test_df['patient_id'].unique())
    assert len(train_pids_actual & val_pids_actual)  == 0, "Train/Val patient overlap!"
    assert len(train_pids_actual & test_pids_actual) == 0, "Train/Test patient overlap!"
    assert len(val_pids_actual   & test_pids_actual) == 0, "Val/Test patient overlap!"
    print("  ✓ Patient disjointness verified")

    for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        dist = df['label'].value_counts().to_dict()
        print(f"{name}: {len(df)} images")
        for cls in CLASS_NAMES:
            cnt = dist.get(cls, 0)
            pct = 100 * cnt / max(len(df), 1)
            print(f"    {cls}: {cnt} ({pct:.1f}%)")

    return train_df, val_df, test_df

# ════════════════════════════════════════════════════════════
# 13: 5-FOLD GROUP CROSS-VALIDATION MAIN
# ════════════════════════════════════════════════════════════
def main():
    from sklearn.model_selection import StratifiedGroupKFold
    import datetime

    start = time.time()
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    exp_tag = 'frozen' if FREEZE_BACKBONE else 'partial_unfreeze'

    BASE_OUTPUT = OUTPUT_DIR
    os.makedirs(BASE_OUTPUT, exist_ok=True)

    print("╔══════════════════════════════════════════════════════════════╗")
    print("║  Bladder Classification — v3 5-Fold Group CV               ║")
    print(f"║  Mode: {'FROZEN backbone                              ' if FREEZE_BACKBONE else 'PARTIAL UNFREEZE (DINOv2 last2 + DenseBlock4)    '}║")
    print("║  Teacher-only (no student distillation in CV)              ║")
    print("║  StratifiedGroupKFold for class-balanced test folds        ║")
    print("╚══════════════════════════════════════════════════════════════╝")

    # ── Load all patients from annotations_fixed.csv ──
    scan_for_images()
    df_orig = pd.read_csv(ANNOTATIONS_CSV)
    df_orig.columns = df_orig.columns.str.strip()
    filename_col = 'HLY' if 'HLY' in df_orig.columns else df_orig.columns[0]
    df_orig['patient_id'] = df_orig[filename_col].apply(extract_patient_id)
    df_orig['label'] = df_orig['tissue type'].map(LABEL_MAP)
    df_orig['path'] = df_orig.apply(
        lambda row: IMAGE_PATH_INDEX.get(str(row[filename_col]).strip().lower()), axis=1)
    df_orig = df_orig[df_orig['path'].notna()].copy()

    all_patients = sorted(df_orig['patient_id'].unique().tolist())
    print(f"\nAll {len(all_patients)} patients: {all_patients}")
    print(f"Output base: {BASE_OUTPUT}")

    # ── StratifiedGroupKFold on image-level labels + patient groups ──
    X = np.zeros(len(df_orig))
    y = df_orig['label'].values
    groups = df_orig['patient_id'].values

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

    fold_results = []
    all_preds, all_labels, all_pids = [], [], []

    for fold_idx, (train_val_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):

        fold_dir = os.path.join(BASE_OUTPUT, f'fold_{fold_idx+1}')
        os.makedirs(fold_dir, exist_ok=True)

        test_pids  = sorted(set(groups[test_idx]))
        train_val_pids = sorted(set(groups[train_val_idx]))

        # ── Smart val selection ──────────────────────────────────
        # Pick 1-2 patients as val such that:
        #   (a) they have >= 30 images
        #   (b) they cover >= 2 of the 3 classes
        #   (c) they are NOT the largest patients (keep those for training)
        pid_stats = df_orig[df_orig['patient_id'].isin(train_val_pids)].groupby('patient_id').agg(
            n_images=('patient_id', 'count'),
            n_classes=('label', 'nunique')
        ).reset_index()

        # Candidate val patients: at least 30 images and 2+ classes
        candidates = pid_stats[(pid_stats['n_images'] >= 30) & (pid_stats['n_classes'] >= 2)]
        if len(candidates) == 0:
            # Fallback: just pick the one with most classes
            candidates = pid_stats[pid_stats['n_classes'] == pid_stats['n_classes'].max()]

        # Pick the candidate with the median image count (not biggest, not smallest)
        candidates = candidates.sort_values('n_images')
        val_pick_idx = len(candidates) // 2
        val_pid = [int(candidates.iloc[val_pick_idx]['patient_id'])]

        # If val set is still small (< 30 images), pick a second candidate
        val_n = pid_stats[pid_stats['patient_id'].isin(val_pid)]['n_images'].sum()
        if val_n < 30 and len(candidates) > 1:
            remaining = candidates[~candidates['patient_id'].isin(val_pid)]
            if len(remaining) > 0:
                second = int(remaining.iloc[len(remaining)//2]['patient_id'])
                val_pid.append(second)

        train_pids = [p for p in train_val_pids if p not in val_pid]

        print(f"\n{'═'*60}")
        print(f"  FOLD {fold_idx+1}/5")
        print(f"{'═'*60}")

        # Load backbones fresh each fold
        global dino_model, dense_model, feat_dim
        feat_dim = load_backbones()

        train_df, val_df, test_df = load_data_for_fold(
            fold_idx+1, train_pids, test_pids, val_pids=val_pid)

        norm = fit_normalizer(train_df.sample(min(len(train_df), 200), random_state=42))

        print("" + "=" * 60)
        print(f"FOLD {fold_idx+1} — EXTRACTING FEATURES")
        print("=" * 60)

        train_data = extract_features_for_split(train_df, f"Fold{fold_idx+1} Train", norm, use_cache=True)
        val_data   = extract_features_for_split(val_df,   f"Fold{fold_idx+1} Val",   norm, use_cache=True)
        val_tta    = extract_features_with_tta(val_df, norm, f"Fold{fold_idx+1} Val TTA")
        test_tta   = extract_features_with_tta(test_df, norm, f"Fold{fold_idx+1} Test TTA")

        class_weights = compute_class_weights(train_data)

        # Train teacher only
        teacher = train_teacher(train_data, val_data, class_weights)

        # Evaluate
        val_preds,  val_labels,  _ = evaluate_full([teacher], val_tta,  f"Fold{fold_idx+1} Val")
        test_preds, test_labels, _ = evaluate_full([teacher], test_tta, f"Fold{fold_idx+1} Test")

        val_metrics  = print_results(val_preds,  val_labels,  f"FOLD {fold_idx+1} VAL")
        test_metrics = print_results(test_preds, test_labels, f"FOLD {fold_idx+1} TEST")

        # Per-patient test breakdown
        test_pids_list = [item['patient'] for item in test_tta if item is not None]
        print(f"\n  Per-patient breakdown (Fold {fold_idx+1} Test):")
        for pid in sorted(set(test_pids_list)):
            idxs  = [i for i, p in enumerate(test_pids_list) if p == pid]
            p_lab = [test_labels[i] for i in idxs]
            p_prd = [test_preds[i]  for i in idxs]
            p_acc = accuracy_score(p_lab, p_prd)
            dist  = Counter(p_lab)
            dist_str = ', '.join(f"{IDX_TO_CLASS[k]}={v}" for k, v in sorted(dist.items()))
            print(f"    Patient {pid}: {len(idxs)} imgs ({dist_str}) → acc={p_acc*100:.1f}%")

        # Save artifacts
        save_cm(test_preds, test_labels,
                os.path.join(fold_dir, f'cm_test_fold{fold_idx+1}.png'),
                f"Fold {fold_idx+1} Test — {exp_tag}")
        save_cm(val_preds, val_labels,
                os.path.join(fold_dir, f'cm_val_fold{fold_idx+1}.png'),
                f"Fold {fold_idx+1} Val — {exp_tag}")
        torch.save(teacher.state_dict(),
                   os.path.join(fold_dir, f'teacher_fold{fold_idx+1}.pt'))

        fold_results.append({
            'fold': fold_idx + 1,
            'train_pids': train_pids,
            'val_pids': val_pid,
            'test_pids': test_pids,
            'val_metrics': val_metrics,
            'test_metrics': test_metrics,
        })
        all_preds.extend(test_preds)
        all_labels.extend(test_labels)
        all_pids.extend(test_pids_list)

        # Free GPU memory
        del dino_model, dense_model, teacher
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # ════════════════════════════════════════════════════════
    # AGGREGATE RESULTS
    # ════════════════════════════════════════════════════════
    print(f"\n{'═'*60}")
    print("  5-FOLD CV — AGGREGATE RESULTS")
    print(f"{'═'*60}")
    print(f"  Backbone: {exp_tag}")
    print(f"  Fold splitter: StratifiedGroupKFold")
    print()

    accs  = [r['test_metrics']['accuracy']          for r in fold_results]
    bals  = [r['test_metrics']['balanced_accuracy'] for r in fold_results]
    hgcrs = [r['test_metrics']['hgc_recall']        for r in fold_results]

    print(f"  Per-fold Test Accuracy:          {' | '.join(f'{a*100:.1f}%' for a in accs)}")
    print(f"  Mean ± Std:   Acc={np.mean(accs)*100:.1f}% ± {np.std(accs)*100:.1f}%")
    print(f"                Bal={np.mean(bals)*100:.1f}% ± {np.std(bals)*100:.1f}%")
    print(f"                HGC-R={np.mean(hgcrs)*100:.1f}% ± {np.std(hgcrs)*100:.1f}%")

    # Grand confusion matrix
    save_cm(all_preds, all_labels,
            os.path.join(BASE_OUTPUT, f'cm_grand_{exp_tag}.png'),
            f"Grand CV — {exp_tag}")
    grand_metrics = print_results(all_preds, all_labels, "GRAND CV TEST")

    # Per-patient grand summary
    print("\n  Per-patient grand test accuracy:")
    for pid in sorted(set(all_pids)):
        idxs  = [i for i, p in enumerate(all_pids) if p == pid]
        p_lab = [all_labels[i] for i in idxs]
        p_prd = [all_preds[i]  for i in idxs]
        p_acc = accuracy_score(p_lab, p_prd)
        dist  = Counter(p_lab)
        dist_str = ', '.join(f"{IDX_TO_CLASS[k]}={v}" for k, v in sorted(dist.items()))
        print(f"    Patient {pid}: ({dist_str}) → acc={p_acc*100:.1f}%")

    elapsed = time.time() - start
    results_out = {
        'experiment': f'v3_5foldcv_{exp_tag}',
        'timestamp': timestamp,
        'runtime_minutes': elapsed / 60,
        'backbone': exp_tag,
        'fold_splitter': 'StratifiedGroupKFold',
        'n_folds': 5,
        'fold_results': fold_results,
        'grand_metrics': grand_metrics,
        'summary': {
            'acc_mean':  float(np.mean(accs)),
            'acc_std':   float(np.std(accs)),
            'bal_mean':  float(np.mean(bals)),
            'bal_std':   float(np.std(bals)),
            'hgcr_mean': float(np.mean(hgcrs)),
            'hgcr_std':  float(np.std(hgcrs)),
        }
    }
    results_path = os.path.join(BASE_OUTPUT, f'cv_results_{exp_tag}_{timestamp}.json')
    with open(results_path, 'w') as f:
        json.dump(results_out, f, indent=2, default=str)

    print(f"\n{'═'*60}")
    print(f"  5-FOLD CV COMPLETE — Runtime: {elapsed/60:.1f} min")
    print(f"  Acc: {np.mean(accs)*100:.1f}% ± {np.std(accs)*100:.1f}%  |  "
          f"Bal: {np.mean(bals)*100:.1f}% ± {np.std(bals)*100:.1f}%  |  "
          f"HGC-R: {np.mean(hgcrs)*100:.1f}% ± {np.std(hgcrs)*100:.1f}%")
    print(f"  Results saved → {results_path}")
    print(f"{'═'*60}")


if __name__ == '__main__':
    main()
main()


╔══════════════════════════════════════════════════════════════╗
║  Bladder Classification — v3 5-Fold Group CV               ║
║  Mode: FROZEN backbone                              ║
║  Teacher-only (no student distillation in CV)              ║
║  StratifiedGroupKFold for class-balanced test folds        ║
╚══════════════════════════════════════════════════════════════╝
SCANNING FILESYSTEM
  Indexed 3472 images

All 22 patients: [0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 21, 22, 23, 24, 25]
Output base: /teamspace/studios/this_studio/output_5foldcv_frozen

════════════════════════════════════════════════════════════
  FOLD 1/5
════════════════════════════════════════════════════════════
LOADING BACKBONES — FROZEN
  Loading DINOv2...


Using cache found in /home/zeus/.cache/torch/hub/facebookresearch_dinov2_main


  ✓ dinov2_vitb14 — FULLY FROZEN, dim=768
  ✓ DenseNet121 — FULLY FROZEN, dim=1024
  Total feature dim: 1792
SCANNING FILESYSTEM
  Indexed 3472 images
FOLD 1 — LOADING DATA
  Train patients: [np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(11), np.int64(16), np.int64(18), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]
  Test  patients: [np.int64(0), np.int64(10), np.int64(13), np.int64(14), np.int64(17)]
  Val patients:   [12]
  Val WLI-only filter: kept 84 / 113
  ✓ Patient disjointness verified
Train: 11241 images
    HGC: 3249 (28.9%)
    LGC: 4077 (36.3%)
    Normal: 3915 (34.8%)
Val: 84 images
    HGC: 0 (0.0%)
    LGC: 21 (25.0%)
    Normal: 63 (75.0%)
Test: 355 images
    HGC: 108 (30.4%)
    LGC: 144 (40.6%)
    Normal: 103 (29.0%)
  Fitting LAB normalizer...
  ✓ Normalizer fitted on 50 images
FOLD 1 — EXTRACTING FEATURES


Fold1 Train:   0%|          | 0/11241 [00:00<?, ?it/s]

  ⚡ Cache hits: 455/11241
  ✓ Extracted 11241 bags


...

============================================================FOLD 2 VAL============================================================
  Accuracy:          75.00%
  Balanced Accuracy: 50.00%
              precision    recall  f1-score   support

         HGC     0.0000    0.0000    0.0000         0
         LGC     0.0000    0.0000    0.0000        21
      Normal     0.9844    1.0000    0.9921        63

    accuracy                         0.7500        84
   macro avg     0.3281    0.3333    0.3307        84
weighted avg     0.7383    0.7500    0.7441        84

  HGC: Recall=0.00%, Precision=0.00%
Confusion Matrix:                HGC       LGC    Normal
         HGC        0         0         0
         LGC       20         0         1
      Normal        0         0        63
Target acc≥90%:    ❌ (75.0%)
  Target HGC R≥95%:  ❌ (0.0%)
============================================================FOLD 2 TEST============================================================
  Accuracy:          

Using cache found in /home/zeus/.cache/torch/hub/facebookresearch_dinov2_main


  ✓ dinov2_vitb14 — FULLY FROZEN, dim=768
  ✓ DenseNet121 — FULLY FROZEN, dim=1024
  Total feature dim: 1792
SCANNING FILESYSTEM
  Indexed 3472 images
FOLD 3 — LOADING DATA
  Train patients: [np.int64(0), np.int64(1), np.int64(4), np.int64(6), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(13), np.int64(14), np.int64(17), np.int64(18), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25)]
  Test  patients: [np.int64(2), np.int64(5), np.int64(7), np.int64(16)]
  Val patients:   [12]
  Val WLI-only filter: kept 84 / 113
  ✓ Patient disjointness verified
Train: 10656 images
    HGC: 3465 (32.5%)
    LGC: 3582 (33.6%)
    Normal: 3609 (33.9%)
Val: 84 images
    HGC: 0 (0.0%)
    LGC: 21 (25.0%)
    Normal: 63 (75.0%)
Test: 420 images
    HGC: 84 (20.0%)
    LGC: 199 (47.4%)
    Normal: 137 (32.6%)
  Fitting LAB normalizer...
  ✓ Normalizer fitted on 50 images
FOLD 3 — EXTRACTING FEATURES


Fold3 Train:   0%|          | 0/10656 [00:00<?, ?it/s]

: 